# Validate ResStock loads against EIA-861

This notebook loads ResStock building metadata, utility
assignments, and **`_sb` annual load curves** for CT using the `res_2024_amy2018_2_sb` release, then
compares weighted residential totals to EIA-861 utility sales — first on customer
counts, then on kWh after adjusting for customer-count mismatch.

---

## How to use

1. Set `STATE` (lowercase 2-letter abbreviation) and `RESSTOCK_RELEASE` in Parameters.
2. Optionally set `UPGRADE` (default `"00"`) and `EIA_YEAR` (default `2018`, to match
   ResStock AMY 2018).
3. Restart the kernel and run all cells.

---

## Data sources

| Input | S3 location |
|-------|-------------|
| Metadata (`metadata-sb`) | `s3://data.sb/nrel/resstock/<release>_sb/metadata/state=<ST>/upgrade=<uu>/metadata-sb.parquet` |
| Utility assignment | `s3://data.sb/nrel/resstock/<release>_sb/metadata_utility/state=<ST>/utility_assignment.parquet` |
| Annual load curves (`_sb`) | `s3://data.sb/nrel/resstock/<release>_sb/load_curve_annual/state=<ST>/upgrade=<uu>/<ST>_upgrade<uu>_metadata_and_annual_results.parquet` |
| EIA-861 utility stats | `s3://data.sb/eia/861/electric_utility_stats/year=<year>/state=<ST>/data.parquet` |

All ResStock inputs for this notebook come from the **`_sb`** companion release
(Switchbox-modified metadata and load curves). We use the **`_sb`** `load_curve_annual`
file, not the raw ResStock one: the `_sb` annual values reflect post-processing
adjustments (e.g. MF non-HVAC scaling) and match the sum of the `_sb` monthly load
curves, whereas the raw annual file does not. Annual electricity is read directly from
`out.electricity.total.energy_consumption.kwh`; sample `weight` comes from `metadata-sb`.

---

## Notebook sections

| Section | What it covers |
|---------|---------------|
| **1 — Load data** | Read metadata, utility assignment, and `_sb` annual loads; keep the `bldg_id` intersection |
| **2 — Sum by utility** | Weighted electricity totals by `sb.electric_utility` |
| **3 — Compare to EIA** | Customer-count ratios; then kWh comparison after scaling ResStock to EIA customer counts |
| **4 — Assumptions** | *(todo)* Document limitations and interpretation guidance |

## Parameters

Change `STATE` and `RESSTOCK_RELEASE` here. Everything else derives from these values.

In [1]:
from __future__ import annotations

from typing import cast

import polars as pl
from IPython.display import display

In [2]:
# ── Change these to validate a different state or release ─────────────────────
STATE = "ct"  # lowercase state abbreviation, e.g. "ri", "ny", "ma"
RESSTOCK_RELEASE = "res_2024_amy2018_2_sb"  # _sb release name
UPGRADE = "00"  # zero-padded ResStock upgrade id
EIA_YEAR = 2018  # EIA-861 report year (2018 aligns with ResStock AMY 2018)
# ──────────────────────────────────────────────────────────────────────────────

STATE_UPPER = STATE.upper()

# All ResStock inputs for this notebook come from the _sb companion release.
RESSTOCK_RELEASE_SB = RESSTOCK_RELEASE if RESSTOCK_RELEASE.endswith("_sb") else f"{RESSTOCK_RELEASE}_sb"

S3_RESSTOCK = "s3://data.sb/nrel/resstock"
S3_EIA861 = "s3://data.sb/eia/861/electric_utility_stats"

BLDG_ID = "bldg_id"
UTILITY_COL = "sb.electric_utility"
WEIGHT_COL = "weight"
# Annual _sb electricity total (note the ".kwh" suffix on the annual file)
ANNUAL_ELEC_COL = "out.electricity.total.energy_consumption.kwh"

PATH_METADATA = (
    f"{S3_RESSTOCK}/{RESSTOCK_RELEASE_SB}/metadata/state={STATE_UPPER}/upgrade={UPGRADE}/metadata-sb.parquet"
)
PATH_UTILITY_ASSIGNMENT = (
    f"{S3_RESSTOCK}/{RESSTOCK_RELEASE_SB}/metadata_utility/state={STATE_UPPER}/utility_assignment.parquet"
)
PATH_ANNUAL = (
    f"{S3_RESSTOCK}/{RESSTOCK_RELEASE_SB}/load_curve_annual/"
    f"state={STATE_UPPER}/upgrade={UPGRADE}/"
    f"{STATE_UPPER}_upgrade{UPGRADE}_metadata_and_annual_results.parquet"
)
PATH_EIA861 = f"{S3_EIA861}/year={EIA_YEAR}/state={STATE_UPPER}/data.parquet"

print(f"State:              {STATE_UPPER}")
print(f"ResStock (_sb):     {RESSTOCK_RELEASE_SB}")
print(f"Upgrade:            {UPGRADE}")
print(f"EIA-861 year:       {EIA_YEAR}")
print()
print(f"Metadata:           {PATH_METADATA}")
print(f"Utility assignment: {PATH_UTILITY_ASSIGNMENT}")
print(f"Annual load curves: {PATH_ANNUAL}")
print(f"EIA-861 stats:      {PATH_EIA861}")

State:              CT
ResStock (_sb):     res_2024_amy2018_2_sb
Upgrade:            00
EIA-861 year:       2018

Metadata:           s3://data.sb/nrel/resstock/res_2024_amy2018_2_sb/metadata/state=CT/upgrade=00/metadata-sb.parquet
Utility assignment: s3://data.sb/nrel/resstock/res_2024_amy2018_2_sb/metadata_utility/state=CT/utility_assignment.parquet
Annual load curves: s3://data.sb/nrel/resstock/res_2024_amy2018_2_sb/load_curve_annual/state=CT/upgrade=00/CT_upgrade00_metadata_and_annual_results.parquet
EIA-861 stats:      s3://data.sb/eia/861/electric_utility_stats/year=2018/state=CT/data.parquet


## Section 1: Load data

We read three ResStock tables from the **`_sb`** release on S3:

1. **`metadata-sb.parquet`** — building attributes and sample `weight` (one row per
   `bldg_id` for this upgrade).
2. **`utility_assignment.parquet`** — `sb.electric_utility` / `sb.gas_utility` per building.
3. **`load_curve_annual`** — a single parquet with one row per `bldg_id` carrying
   `out.electricity.total.energy_consumption.kwh` (the annual electricity total). We
   read this directly as `annual_kwh` — no monthly aggregation needed.

After loading, we print shape, schema, and a small sample so the reader can confirm
the expected columns are present before aggregation.

> **Note:** The `_sb` `load_curve_annual` values already reflect Switchbox's
> post-processing adjustments (they match the sum of the `_sb` monthly load curves,
> and differ from the raw ResStock annual file). Reading the single annual parquet is
> far faster than fetching thousands of monthly files.

In [3]:
def load_parquet(path: str) -> pl.DataFrame:
    """Collect a parquet file (or Hive directory) from S3 or local disk."""
    return cast(pl.DataFrame, pl.scan_parquet(path).collect())


def preview(name: str, df: pl.DataFrame, key_cols: list[str] | None = None) -> None:
    """Print shape, optional key-column check, schema head, and a few rows."""
    print(f"=== {name} ===")
    print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} cols")
    if key_cols:
        missing = [c for c in key_cols if c not in df.columns]
        if missing:
            raise ValueError(f"{name} is missing required columns: {missing}")
        print(f"required columns present: {key_cols}")
    print("schema (first 25):")
    for col, dtype in list(df.schema.items())[:25]:
        print(f"  {col}: {dtype}")
    if len(df.schema) > 25:
        print(f"  … ({len(df.schema) - 25} more columns)")
    display(df.head(5))
    print()

In [4]:
print(f"Loading metadata from:\n  {PATH_METADATA}\n")
metadata = load_parquet(PATH_METADATA)
preview("metadata-sb", metadata, key_cols=[BLDG_ID, WEIGHT_COL])

Loading metadata from:
  s3://data.sb/nrel/resstock/res_2024_amy2018_2_sb/metadata/state=CT/upgrade=00/metadata-sb.parquet



=== metadata-sb ===
shape: 6,166 rows x 188 cols
required columns present: ['bldg_id', 'weight']
schema (first 25):
  upgrade: Int64
  weight: Float64
  in.sqft: Int64
  in.representative_income: Float64
  in.ahs_region: String
  in.aiannh_area: String
  in.area_median_income: String
  in.ashrae_iecc_climate_zone_2004: String
  in.ashrae_iecc_climate_zone_2004_2_a_split: String
  in.bathroom_spot_vent_hour: String
  in.battery: String
  in.bedrooms: String
  in.building_america_climate_zone: String
  in.cec_climate_zone: String
  in.ceiling_fan: String
  in.census_division: String
  in.census_division_recs: String
  in.census_region: String
  in.city: String
  in.clothes_dryer: String
  in.clothes_dryer_usage_level: String
  in.clothes_washer: String
  in.clothes_washer_presence: String
  in.clothes_washer_usage_level: String
  in.cooking_range: String
  … (163 more columns)


upgrade,weight,in.sqft,in.representative_income,in.ahs_region,in.aiannh_area,in.area_median_income,in.ashrae_iecc_climate_zone_2004,in.ashrae_iecc_climate_zone_2004_2_a_split,in.bathroom_spot_vent_hour,in.battery,in.bedrooms,in.building_america_climate_zone,in.cec_climate_zone,in.ceiling_fan,in.census_division,in.census_division_recs,in.census_region,in.city,in.clothes_dryer,in.clothes_dryer_usage_level,in.clothes_washer,in.clothes_washer_presence,in.clothes_washer_usage_level,in.cooking_range,in.cooking_range_usage_level,in.cooling_setpoint,in.cooling_setpoint_has_offset,in.cooling_setpoint_offset_magnitude,in.cooling_setpoint_offset_period,in.corridor,in.county,in.county_and_puma,in.county_name,in.dehumidifier,in.dishwasher,in.dishwasher_usage_level,…,in.solar_hot_water,in.state,in.tenure,in.units_represented,in.usage_level,in.utility_bill_electricity_fixed_charges,in.utility_bill_electricity_marginal_rates,in.utility_bill_fuel_oil_fixed_charges,in.utility_bill_fuel_oil_marginal_rates,in.utility_bill_natural_gas_fixed_charges,in.utility_bill_natural_gas_marginal_rates,in.utility_bill_propane_fixed_charges,in.utility_bill_propane_marginal_rates,in.utility_bill_scenario_names,in.utility_bill_simple_filepaths,in.vacancy_status,in.vintage,in.vintage_acs,in.water_heater_efficiency,in.water_heater_fuel,in.water_heater_in_unit,in.water_heater_location,in.weather_file_city,in.weather_file_latitude,in.weather_file_longitude,in.window_areas,in.windows,bldg_id,postprocess_group.has_hp,postprocess_group.heating_type,postprocess_group.heating_type_v2,heats_with_electricity,heats_with_natgas,heats_with_oil,heats_with_propane,has_natgas_connection,mf_non_hvac_electricity_adjusted
i64,f64,i64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,i64,str,i64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,i64,bool,str,str,bool,bool,bool,bool,bool,bool
0,252.301639,1207,47457.0,"""Non-CBSA New England""","""No""","""30-60%""","""5A""","""5A""","""Hour4""","""None""","""3""","""Cold""","""None""","""Standard Efficiency""","""New England""","""New England""","""Northeast""","""CT, Norwalk""","""Electric""","""100% Usage""","""Standard""","""Yes""","""100% Usage""","""Electric Resistance""","""100% Usage""","""72F""","""No""","""0F""","""None""","""Not Applicable""","""G0900010""","""G0900010, G09000103""","""Fairfield County""","""None""","""None""","""100% Usage""",…,"""None""","""CT""","""Renter""",1,"""Medium""",10,0.217925,"""0""","""3.012653846""","""11.25""","""1.397604181""","""0""","""3.365153846""","""Utility Rates - Fixed + Variab…","""data/simple_rates/State.tsv""","""Occupied""","""1980s""","""1980-99""","""Propane Standard""","""Propane""","""Yes""","""Living Space""","""Bridgeport Igor I""",41.18,-73.15,"""F15 B15 L15 R15""","""Single, Clear, Non-metal""",504616,false,"""fossil_fuel""","""delivered_fuels""",false,false,false,true,false,false
0,252.301639,881,149501.0,"""Non-CBSA New England""","""No""","""150%+""","""5A""","""5A""","""Hour17""","""None""","""2""","""Cold""","""None""","""Standard Efficiency""","""New England""","""New England""","""Northeast""","""Not in a census Place""","""Electric""","""80% Usage""","""EnergyStar""","""Yes""","""80% Usage""","""Electric Resistance""","""80% Usage""","""60F""","""Yes""","""2F""","""Night Setup +2h""","""Not Applicable""","""G0900090""","""G0900090, G09000903""","""New Haven County""","""None""","""290 Rated kWh""","""80% Usage""",…,"""None""","""CT""","""Owner""",1,"""Low""",10,0.217925,"""0""","""3.012653846""","""11.25""","""1.397604181""","""0""","""3.365153846""","""Utility Rates - Fixed + Variab…","""data/simple_rates/State.tsv""","""Occupied""","""1950s""","""1940-59""","""Fuel Oil Standard""","""Fuel Oil""","""Yes""","""Living Space""","""Meriden Markham Muni""",41.51,-72.83,"""F9 B9 L9 R9""","""Double, Clear, Non-metal, Air,…",31933,false,""

In [5]:
print(f"Loading utility assignment from:\n  {PATH_UTILITY_ASSIGNMENT}\n")
utility_assignment = load_parquet(PATH_UTILITY_ASSIGNMENT)
preview(
    "utility_assignment",
    utility_assignment,
    key_cols=[BLDG_ID, UTILITY_COL],
)

Loading utility assignment from:
  s3://data.sb/nrel/resstock/res_2024_amy2018_2_sb/metadata_utility/state=CT/utility_assignment.parquet

=== utility_assignment ===
shape: 6,166 rows x 3 cols
required columns present: ['bldg_id', 'sb.electric_utility']
schema (first 25):
  bldg_id: Int64
  sb.electric_utility: String
  sb.gas_utility: String


bldg_id,sb.electric_utility,sb.gas_utility
i64,str,str
504616,"""ct_eversource""",null
31933,"""ct_eversource""","""yankee_gas"""
338527,"""ct_eversource""",null
341661,"""ct_eversource""",null
423800,"""ct_eversource""","""ct_natural_gas"""


In [6]:
print(f"Loading annual load curves from _sb:\n  {PATH_ANNUAL}\n")
annual = load_parquet(PATH_ANNUAL).select(
    BLDG_ID,
    pl.col(ANNUAL_ELEC_COL).alias("annual_kwh"),
)
preview("load_curve_annual (_sb)", annual, key_cols=[BLDG_ID, "annual_kwh"])

Loading annual load curves from _sb:
  s3://data.sb/nrel/resstock/res_2024_amy2018_2_sb/load_curve_annual/state=CT/upgrade=00/CT_upgrade00_metadata_and_annual_results.parquet

=== load_curve_annual (_sb) ===
shape: 6,166 rows x 2 cols
required columns present: ['bldg_id', 'annual_kwh']
schema (first 25):
  bldg_id: Int64
  annual_kwh: Float64


bldg_id,annual_kwh
i64,f64
100164,298.914228
100771,25613.693
100957,5034.167
101537,4664.631
101091,8165.259


### Align on shared buildings

Metadata, utility assignment, and `_sb` annual loads should each have one row per
building for this state/upgrade. Small mismatches can occur when a building is present
in one table but not another (the `_sb` annual file, for example, has one fewer row
than metadata for CT).

For the rest of this notebook we keep only the **intersection** of `bldg_id`s across
all three tables, and report any IDs that were dropped.

In [7]:
n_meta = metadata.height
n_ua = utility_assignment.height
n_annual = annual.height
n_meta_ids = metadata[BLDG_ID].n_unique()
n_ua_ids = utility_assignment[BLDG_ID].n_unique()
n_annual_ids = annual[BLDG_ID].n_unique()

print("Before intersection:")
print(f"  metadata:           {n_meta:,} rows, {n_meta_ids:,} unique {BLDG_ID}")
print(f"  utility_assignment: {n_ua:,} rows, {n_ua_ids:,} unique {BLDG_ID}")
print(f"  annual (_sb):       {n_annual:,} rows, {n_annual_ids:,} unique {BLDG_ID}")

if n_meta != n_meta_ids:
    print("WARNING: metadata has duplicate bldg_id values")
if n_ua != n_ua_ids:
    print("WARNING: utility_assignment has duplicate bldg_id values")
if n_annual != n_annual_ids:
    print("WARNING: annual (_sb) has duplicate bldg_id values")

meta_ids = set(metadata[BLDG_ID].to_list())
ua_ids = set(utility_assignment[BLDG_ID].to_list())
annual_ids = set(annual[BLDG_ID].to_list())
shared_ids = meta_ids & ua_ids & annual_ids

dropped_meta = sorted(meta_ids - shared_ids)
dropped_ua = sorted(ua_ids - shared_ids)
dropped_annual = sorted(annual_ids - shared_ids)

print(f"\nShared bldg_ids (metadata ∩ utility_assignment ∩ annual): {len(shared_ids):,}")
print(f"  dropped from metadata only:           {len(dropped_meta):,} {dropped_meta[:10]}")
print(f"  dropped from utility_assignment only: {len(dropped_ua):,} {dropped_ua[:10]}")
print(f"  dropped from annual only:             {len(dropped_annual):,} {dropped_annual[:10]}")

shared_ids_list = list(shared_ids)
metadata = metadata.filter(pl.col(BLDG_ID).is_in(shared_ids_list))
utility_assignment = utility_assignment.filter(pl.col(BLDG_ID).is_in(shared_ids_list))
annual = annual.filter(pl.col(BLDG_ID).is_in(shared_ids_list))

print("\nAfter intersection:")
print(f"  metadata:           {metadata.height:,}")
print(f"  utility_assignment: {utility_assignment.height:,}")
print(f"  annual (_sb):       {annual.height:,}")
assert metadata.height == utility_assignment.height == annual.height
assert metadata[BLDG_ID].n_unique() == metadata.height

Before intersection:
  metadata:           6,166 rows, 6,166 unique bldg_id
  utility_assignment: 6,166 rows, 6,166 unique bldg_id
  annual (_sb):       6,166 rows, 6,166 unique bldg_id

Shared bldg_ids (metadata ∩ utility_assignment ∩ annual): 6,166
  dropped from metadata only:           0 []
  dropped from utility_assignment only: 0 []
  dropped from annual only:             0 []

After intersection:
  metadata:           6,166
  utility_assignment: 6,166
  annual (_sb):       6,166


## Section 2: Sum electricity by utility

Join the aligned annual loads to utility assignment and metadata `weight` on
`bldg_id`, then for each `sb.electric_utility` sum **weighted** annual electricity:

$$\text{resstock\_total\_kwh} = \sum_i (\text{annual\_kwh}_i \times \text{weight}_i)$$

ResStock `weight` is the sample expansion factor from `metadata-sb` (each building
represents roughly 252 dwellings). Customer counts are likewise
$\sum_i \text{weight}_i$.

Buildings with a null electric utility assignment are reported separately and
excluded from the per-utility totals.

In [8]:
annual_with_utility = (
    annual.select(BLDG_ID, "annual_kwh")
    .join(
        metadata.select(BLDG_ID, WEIGHT_COL),
        on=BLDG_ID,
        how="inner",
        validate="1:1",
    )
    .join(
        utility_assignment.select(BLDG_ID, UTILITY_COL),
        on=BLDG_ID,
        how="inner",
        validate="1:1",
    )
    .with_columns((pl.col("annual_kwh") * pl.col(WEIGHT_COL)).alias("weighted_kwh"))
)

n_null_utility = annual_with_utility.filter(pl.col(UTILITY_COL).is_null()).height
if n_null_utility:
    print(f"WARNING: {n_null_utility:,} buildings have null {UTILITY_COL}; excluded from totals")

resstock_by_utility = (
    annual_with_utility.filter(pl.col(UTILITY_COL).is_not_null())
    .group_by(UTILITY_COL)
    .agg(
        pl.len().alias("buildings"),
        pl.col(WEIGHT_COL).sum().alias("resstock_customers"),
        pl.col("weighted_kwh").sum().alias("resstock_total_kwh"),
    )
    .sort("resstock_total_kwh", descending=True)
    .rename({UTILITY_COL: "utility_code"})
)

resstock_by_utility_with_total = pl.concat(
    [
        resstock_by_utility,
        pl.DataFrame(
            {
                "utility_code": ["**TOTAL**"],
                "buildings": [resstock_by_utility["buildings"].sum()],
                "resstock_customers": [resstock_by_utility["resstock_customers"].sum()],
                "resstock_total_kwh": [resstock_by_utility["resstock_total_kwh"].sum()],
            },
            schema=resstock_by_utility.schema,
        ),
    ]
)

print(f"ResStock weighted electricity by {UTILITY_COL} ({STATE_UPPER}, upgrade {UPGRADE}, _sb):")
display(resstock_by_utility_with_total)

ResStock weighted electricity by sb.electric_utility (CT, upgrade 00, _sb):


utility_code,buildings,resstock_customers,resstock_total_kwh
str,u32,f64,f64
"""ct_eversource""",3539,892895.499466,7.7532e9
"""ct_ui""",1436,362305.153217,2.9652e9
"""frp""",810,204364.327372,1.6569e9
"""mohegan_tribal""",216,54497.153966,4.7302e8
"""norwich_muni""",56,14128.891769,1.2174e8
…,…,…,…
"""south_norwalk_muni""",25,6307.540968,5.3700e7
"""jewett_muni""",3,756.904916,6.9608e6
"""bozrah_muni""",3,756.904916,5.8675e6


## Section 3: Compare to EIA-861

We compare ResStock (`_sb`) to **EIA-861 residential sales** in two steps:

1. **Customer counts** — `resstock_customers` (sum of sample weights) vs
   `eia_residential_customers`, and their ratio.
2. **Electricity (kWh)** — scale ResStock total kWh by the inverse customer ratio
   so the kWh comparison is not driven by customer-count mismatch, then report
   ratios and % differences.

The join key is a `utility_code` (short std_name, e.g. `ct_eversource`) that both
sides share: ResStock's `sb.electric_utility` on one side, and EIA-861's
`utility_code` column on the other. The EIA-861 parquet's `utility_code` is
derived at *fetch time* from `rate-design-platform`'s
[`utils/utility_codes.py`](https://github.com/switchbox-data/rate-design-platform/blob/main/utils/utility_codes.py)
crosswalk (EIA `utility_id_eia` → std_name) — if a utility was added to that
crosswalk *after* the EIA-861 parquet for this state was last fetched,
`utility_code` will be `null` for it even though `utility_id_eia` is present.

To make this notebook robust to a stale/partial `utility_code` column, we build
our own `utility_id_eia → utility_code` map below (mirroring the current
`utils/utility_codes.py`) and use it to fill in any missing values before
aggregating. **This map is state-specific — update it when switching `STATE`.**

In [9]:
print(f"Loading EIA-861 residential sales from:\n  {PATH_EIA861}\n")
eia_raw = load_parquet(PATH_EIA861)
preview(
    "eia861",
    eia_raw,
    key_cols=["utility_id_eia", "utility_code", "residential_sales_mwh", "residential_customers"],
)

n_null_code = eia_raw.filter(pl.col("utility_code").is_null()).height
print(f"Rows with null utility_code: {n_null_code:,} of {eia_raw.height:,}")

Loading EIA-861 residential sales from:
  s3://data.sb/eia/861/electric_utility_stats/year=2018/state=CT/data.parquet

=== eia861 ===
shape: 66 rows x 25 cols
required columns present: ['utility_id_eia', 'utility_code', 'residential_sales_mwh', 'residential_customers']
schema (first 25):
  year: Int32
  state: String
  utility_id_eia: Int64
  utility_code: String
  utility_name: String
  business_model: Categorical
  entity_type: String
  report_date: Date
  total_sales_mwh: Float32
  total_sales_revenue: Float32
  commercial_sales_mwh: Float32
  commercial_sales_revenue: Float32
  commercial_customers: Float32
  industrial_sales_mwh: Float32
  industrial_sales_revenue: Float32
  industrial_customers: Float32
  other_sales_mwh: Float32
  other_sales_revenue: Float32
  other_customers: Float32
  residential_sales_mwh: Float32
  residential_sales_revenue: Float32
  residential_customers: Float32
  transportation_sales_mwh: Float32
  transportation_sales_revenue: Float32
  transportation_

year,state,utility_id_eia,utility_code,utility_name,business_model,entity_type,report_date,total_sales_mwh,total_sales_revenue,commercial_sales_mwh,commercial_sales_revenue,commercial_customers,industrial_sales_mwh,industrial_sales_revenue,industrial_customers,other_sales_mwh,other_sales_revenue,other_customers,residential_sales_mwh,residential_sales_revenue,residential_customers,transportation_sales_mwh,transportation_sales_revenue,transportation_customers
i32,str,i64,str,str,cat,str,date,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
2018,"""CT""",58667,null,"""North American Power and Gas, …","""retail""","""Retail Power Marketer""",2018-01-01,383654.0,4.08982e7,15578.0,1.7794e6,724.0,0.0,0.0,0.0,0.0,0.0,0.0,368076.0,3.91188e7,35609.0,0.0,0.0,0.0
2018,"""CT""",56212,null,"""Ambit Energy Holdings, LLC""","""retail""","""Retail Power Marketer""",2018-01-01,561659.0,5.0376e7,96024.0,8.882e6,3732.0,0.0,0.0,0.0,0.0,0.0,0.0,465635.0,4.1494e7,47414.0,0.0,0.0,0.0
2018,"""CT""",60025,null,"""Greenbacker Renewable Energy C…","""retail""","""Behind the Meter""",2018-01-01,5690.0,777000.0,824.0,144000.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,4866.0,633000.0,596.0,0.0,0.0,0.0
2018,"""CT""",57487,null,"""XOOM Energy Connecticut, LLC""","""retail""","""Retail Power Marketer""",2018-01-01,73042.0,7.53e6,29968.0,3.2903e6,999.0,0.0,0.0,0.0,0.0,0.0,0.0,43074.0,4.2397e6,5381.0,0.0,0.0,0.0
2018,"""CT""",9734,null,"""City of Jewett City - (CT)""","""retail""","""Municipal""",2018-01-01,23502.0,3.8874e6,8825.0,1.3205e6,293.0,240.0,32100.0,1.0,0.0,0.0,0.0,14437.0,2.5348e6,1899.0,0.0,0.0,0.0



Rows with null utility_code: 66 of 66


### Fill in missing `utility_code` values

The map below is `state`-specific: it lists every `utility_id_eia` we expect for
`STATE` and the `sb.electric_utility` std_name it corresponds to, taken from
`rate-design-platform`'s `utils/utility_codes.py`. We use it only to fill in rows
where `utility_code` is currently `null` — any row that already has a non-null
`utility_code` is left as-is.

**When running this notebook for a different state**, replace this dict with the
`(state=STATE)` entries from `utils/utility_codes.py` (`std_name` →
`eia_utility_ids`), or skip this cell if that state's `utility_code` column is
already fully populated.

In [10]:
# utility_id_eia -> sb.electric_utility std_name, for STATE = "ct".
# Source: rate-design-platform/utils/utility_codes.py (UTILITIES list, state="CT").
EIA_UTILITY_ID_TO_STD_NAME = {
    4176: "ct_eversource",
    19497: "ct_ui",
    6207: "frp",
    13831: "norwich_muni",
    2089: "bozrah_muni",
    9734: "jewett_muni",
    17569: "south_norwalk_muni",
    7716: "groton_muni",
    49826: "mohegan_tribal",
    13825: "norwalk_third_taxing",
    20038: "wallingford_muni",
}

eia_filled = eia_raw.with_columns(
    pl.coalesce(
        pl.col("utility_code"),
        pl.col("utility_id_eia").replace_strict(EIA_UTILITY_ID_TO_STD_NAME, default=None),
    ).alias("utility_code")
)

n_filled = (
    eia_filled.filter(pl.col("utility_code").is_not_null()).height
    - eia_raw.filter(pl.col("utility_code").is_not_null()).height
)
print(f"Filled in utility_code for {n_filled:,} rows using EIA_UTILITY_ID_TO_STD_NAME.")

n_still_null_with_sales = eia_filled.filter(
    pl.col("utility_code").is_null() & (pl.col("residential_sales_mwh") > 0)
).height
if n_still_null_with_sales:
    print(
        f"WARNING: {n_still_null_with_sales:,} utilities with residential sales still have "
        f"no utility_code (not in EIA_UTILITY_ID_TO_STD_NAME) and will be excluded below."
    )

eia_by_utility = (
    eia_filled.filter(pl.col("utility_code").is_not_null())
    .group_by("utility_code")
    .agg(
        (pl.col("residential_sales_mwh").sum() * 1000).alias("eia_residential_kwh"),
        pl.col("residential_customers").sum().alias("eia_residential_customers"),
    )
)

print(f"\nEIA-861 residential sales by utility_code ({STATE_UPPER}, {EIA_YEAR}):")
display(eia_by_utility.sort("eia_residential_kwh", descending=True))

Filled in utility_code for 11 rows using EIA_UTILITY_ID_TO_STD_NAME.

EIA-861 residential sales by utility_code (CT, 2018):


utility_code,eia_residential_kwh,eia_residential_customers
str,f32,f32
"""ct_eversource""",1.0176e10,1.136892e6
"""ct_ui""",2.2013e9,302818.0
"""wallingford_muni""",2.18703008e8,21361.0
"""norwich_muni""",1.28688e8,17756.0
"""groton_muni""",1.0479e8,12096.0
…,…,…
"""norwalk_third_taxing""",2.7819e7,2968.0
"""bozrah_muni""",2.3302e7,2392.0
"""jewett_muni""",1.4437e7,1899.0


### Customer-count comparison

First we join ResStock and EIA on `utility_code` and look **only** at customer
counts. ResStock customers are $\sum_i \text{weight}_i$; EIA customers are
`residential_customers` from EIA-861.

$$\text{customers\_ratio} = \frac{\text{resstock\_customers}}{\text{eia\_residential\_customers}}$$

A ratio near 1 means the sample expansion matches EIA's reported customer base
for that utility. Ratios far from 1 (or `inf` when EIA reports 0 customers)
mean later kWh comparisons must be customer-normalized.

In [11]:
joined = resstock_by_utility.join(eia_by_utility, on="utility_code", how="inner")

n_dropped = resstock_by_utility.height - joined.height
if n_dropped:
    dropped_codes = sorted(set(resstock_by_utility["utility_code"]) - set(joined["utility_code"]))
    print(
        f"WARNING: {n_dropped} ResStock utilities had no EIA-861 match and were "
        f"dropped from the comparison: {dropped_codes}"
    )

customers_comparison = joined.select(
    "utility_code",
    "buildings",
    "resstock_customers",
    "eia_residential_customers",
    (pl.col("resstock_customers") / pl.col("eia_residential_customers")).alias("customers_ratio"),
    (
        (pl.col("resstock_customers") - pl.col("eia_residential_customers")) / pl.col("eia_residential_customers") * 100
    ).alias("customers_pct_diff"),
).sort("resstock_customers", descending=True)

print(f"ResStock vs EIA-861 customer counts by utility ({STATE_UPPER}, {EIA_YEAR}):")
display(customers_comparison)

ResStock vs EIA-861 customer counts by utility (CT, 2018):


utility_code,buildings,resstock_customers,eia_residential_customers,customers_ratio,customers_pct_diff
str,u32,f64,f32,f64,f64
"""ct_eversource""",3539,892895.499466,1.136892e6,0.785383,-21.461713
"""ct_ui""",1436,362305.153217,302818.0,1.196445,19.644524
"""frp""",810,204364.327372,0.0,inf,inf
"""mohegan_tribal""",216,54497.153966,0.0,inf,inf
"""norwich_muni""",56,14128.891769,17756.0,0.795725,-20.427507
…,…,…,…,…,…
"""wallingford_muni""",37,9335.160633,21361.0,0.437019,-56.29811
"""south_norwalk_muni""",25,6307.540968,5519.0,1.142878,14.287751
"""bozrah_muni""",3,756.904916,2392.0,0.316432,-68.356818


### kWh comparison (customer-count normalized)

To isolate differences in **per-customer electricity use**, we scale ResStock
weighted total kWh to EIA's customer count before comparing:

$$
\text{resstock\_total\_kwh\_normalized} =
\text{resstock\_total\_kwh} \times
\frac{\text{eia\_residential\_customers}}{\text{resstock\_customers}}
= \frac{\text{resstock\_total\_kwh}}{\text{customers\_ratio}}
$$

Then:

- `kwh_ratio` = normalized ResStock kWh / EIA residential kWh  
- `kwh_pct_diff` = (normalized ResStock − EIA) / EIA × 100

Utilities with zero EIA customers (or zero ResStock customers) cannot be
normalized this way and are excluded from the kWh table below.

In [12]:
kwh_base = joined.filter(
    (pl.col("eia_residential_customers") > 0) & (pl.col("resstock_customers") > 0) & (pl.col("eia_residential_kwh") > 0)
)

n_excluded_kwh = joined.height - kwh_base.height
if n_excluded_kwh:
    excluded_codes = sorted(set(joined["utility_code"]) - set(kwh_base["utility_code"]))
    print(
        f"Excluded {n_excluded_kwh} utilities from kWh comparison "
        f"(zero EIA/ResStock customers or zero EIA kWh): {excluded_codes}"
    )

kwh_comparison = (
    kwh_base.with_columns(
        (pl.col("resstock_total_kwh") * pl.col("eia_residential_customers") / pl.col("resstock_customers")).alias(
            "resstock_total_kwh_normalized"
        ),
        (pl.col("resstock_customers") / pl.col("eia_residential_customers")).alias("customers_ratio"),
    )
    .with_columns(
        (pl.col("resstock_total_kwh_normalized") / pl.col("eia_residential_kwh")).alias("kwh_ratio"),
        (
            (pl.col("resstock_total_kwh_normalized") - pl.col("eia_residential_kwh"))
            / pl.col("eia_residential_kwh")
            * 100
        ).alias("kwh_pct_diff"),
    )
    .select(
        "utility_code",
        "buildings",
        "resstock_total_kwh",
        "customers_ratio",
        "resstock_total_kwh_normalized",
        "eia_residential_kwh",
        "kwh_ratio",
        "kwh_pct_diff",
    )
    .sort("resstock_total_kwh", descending=True)
)

print(f"ResStock vs EIA-861 residential kWh by utility (customer-count normalized; {STATE_UPPER}, {EIA_YEAR}, _sb):")
display(kwh_comparison)

Excluded 2 utilities from kWh comparison (zero EIA/ResStock customers or zero EIA kWh): ['frp', 'mohegan_tribal']
ResStock vs EIA-861 residential kWh by utility (customer-count normalized; CT, 2018, _sb):


utility_code,buildings,resstock_total_kwh,customers_ratio,resstock_total_kwh_normalized,eia_residential_kwh,kwh_ratio,kwh_pct_diff
str,u32,f64,f64,f64,f32,f64,f64
"""ct_eversource""",3539,7.7532e9,0.785383,9.8719e9,1.0176e10,0.970076,-2.992355
"""ct_ui""",1436,2.9652e9,1.196445,2.4784e9,2.2013e9,1.12587,12.587016
"""norwich_muni""",56,1.2174e8,0.795725,1.5300e8,1.28688e8,1.188913,18.891254
"""norwalk_third_taxing""",39,8.1639e7,3.315284,2.4625e7,2.7819e7,0.885185,-11.48154
"""wallingford_muni""",37,6.7843e7,0.437019,1.5524e8,2.18703008e8,0.709828,-29.017208
"""south_norwalk_muni""",25,5.3700e7,1.142878,4.6987e7,4.1197e7,1.140542,14.054228
"""jewett_muni""",3,6.9608e6,0.398581,1.7464e7,1.4437e7,1.209671,20.967082
"""bozrah_muni""",3,5.8675e6,0.316432,1.8543e7,2.3302e7,0.795761,-20.423946
"""groton_muni""",2,2.2012e6,0.041717,5.2765e7,1.0479e8,0.503535,-49.646489


## Section 4: Assumptions and limitations

*Coming next — document weighting, residential-only EIA scope, year alignment, and
known discrepancy drivers.*